In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
sample = pd.read_csv("../data/sample_submission.csv")

print("Train (số dòng, số cột):", train.shape)
print("Test (số dòng, số cột):", test.shape)
print("Tệp mẫu (số dòng, số cột):", sample.shape)

print("Cột có ở train nhưng không có ở test:",
      sorted(set(train.columns) - set(test.columns)))
print("Các cột của tệp mẫu:", sample.columns.tolist())

display(train.head())
display(sample.head())

In [ ]:
print("Kiểu dữ liệu của từng cột:")
print(train.dtypes.to_string())

print("\nTổng ô thiếu trong train:", train.isna().sum().sum())
print("Tổng ô thiếu trong test:", test.isna().sum().sum())

print("\nID bị trùng trong train:", train["id"].duplicated().sum())
print("ID bị trùng trong test:", test["id"].duplicated().sum())
print("ID của tệp mẫu khớp với test:",
      sample["id"].equals(test["id"]))

In [ ]:
print("Số giao dịch:")
print(train["is_fraud"].value_counts().sort_index())

print("\nTỷ lệ phần trăm:")
print(
    (train["is_fraud"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

In [ ]:
cot_so = [
    "amount_usd",
    "account_balance_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
]

bang_so_sanh = train.groupby("is_fraud")[cot_so].median().round(2).T
display(bang_so_sanh)

plt.figure(figsize=(6, 4))
sns.boxplot(
    data=train,
    x="is_fraud",
    y="hours_since_last_txn",
    showfliers=False
)
plt.title("Thời gian từ giao dịch trước theo nhóm")
plt.xlabel("0 = Bình thường, 1 = Gian lận")
plt.ylabel("Số giờ từ giao dịch trước")
plt.show()

# Lấy 5.000 giao dịch mỗi nhóm để vẽ nhanh hơn
mau_ve = (
    train.groupby("is_fraud", group_keys=False)
    .sample(n=5000, random_state=42)
)

plt.figure(figsize=(6, 4))
sns.boxplot(
    data=mau_ve,
    x="is_fraud",
    y="amount_usd",
    showfliers=False
)
plt.title("Số tiền giao dịch theo nhóm")
plt.xlabel("0 = Bình thường, 1 = Gian lận")
plt.ylabel("Số tiền (USD)")
plt.show()

In [ ]:
bang_danh_muc = (
    train.groupby("merchant_category")["is_fraud"]
    .agg(
        so_giao_dich="size",
        so_gian_lan="sum",
        ty_le_gian_lan="mean"
    )
    .sort_values("ty_le_gian_lan", ascending=False)
)

bang_danh_muc["so_gian_lan"] = bang_danh_muc["so_gian_lan"].astype(int)
bang_danh_muc["ty_le_gian_lan"] = (
    bang_danh_muc["ty_le_gian_lan"] * 100
).round(2)

display(bang_danh_muc)

for cot in ["channel", "auth_method", "device_type", "card_type"]:
    bang = train.groupby(cot)["is_fraud"].agg(
        so_giao_dich="size",
        ty_le_gian_lan="mean"
    )
    bang["ty_le_gian_lan"] = (bang["ty_le_gian_lan"] * 100).round(2)

    print("\nCột:", cot)
    display(bang.sort_values("ty_le_gian_lan", ascending=False))

In [ ]:
for cot in ["time_of_day_hour", "day_of_week"]:
    bang_thoi_gian = (
        train.groupby(cot)["is_fraud"]
        .agg(so_giao_dich="size", ty_le_gian_lan="mean")
        .sort_index()
    )
    bang_thoi_gian["ty_le_gian_lan"] = (
        bang_thoi_gian["ty_le_gian_lan"] * 100
    ).round(2)

    print("\nCột:", cot)
    display(bang_thoi_gian)

In [ ]:
# Tương quan giữa từng cột số và is_fraud
train_so = train.select_dtypes(include=["number", "bool"]).drop(columns=["id"])

tuong_quan = train_so.corr()["is_fraud"].drop("is_fraud")
tuong_quan = tuong_quan.reindex(
    tuong_quan.abs().sort_values(ascending=False).index
)

print("Các cột có tương quan mạnh nhất với is_fraud:")
display(tuong_quan.head(15).round(3))

# Tương quan giữa các cột đầu vào
corr_giua_cac_cot = train_so.drop(columns="is_fraud").corr().abs()
cap_cot = corr_giua_cac_cot.unstack()
cap_cot = cap_cot[
    cap_cot.index.get_level_values(0) < cap_cot.index.get_level_values(1)
]

print("10 cặp cột có tương quan cao nhất:")
display(cap_cot.sort_values(ascending=False).head(10).round(3))

# Phần xu của số tiền: 0 đến 99
phan_xu = ((train["amount_usd"] * 100).round().astype(int) % 100)

bang_phan_xu = (
    train.assign(phan_xu=phan_xu)
    .groupby("phan_xu")["is_fraud"]
    .agg(so_giao_dich="size", ty_le_gian_lan="mean")
)
bang_phan_xu["ty_le_gian_lan"] = (
    bang_phan_xu["ty_le_gian_lan"] * 100
).round(2)

print("15 phần xu xuất hiện nhiều nhất:")
display(bang_phan_xu.sort_values("so_giao_dich", ascending=False).head(15))

# So sánh số tiền tròn đô la với số tiền có phần xu
bang_so_tien_tron = (
    train.assign(so_tien_tron=phan_xu.eq(0))
    .groupby("so_tien_tron")["is_fraud"]
    .agg(so_giao_dich="size", ty_le_gian_lan="mean")
)
bang_so_tien_tron["ty_le_gian_lan"] = (
    bang_so_tien_tron["ty_le_gian_lan"] * 100
).round(2)

print("Số tiền tròn đô la (True) và không tròn (False):")
display(bang_so_tien_tron)

## Kết luận phân tích dữ liệu (EDA)

- Tập train có 699.960 giao dịch và 26 cột; tập test có 299.984 giao dịch và 25 cột. `is_fraud` là cột cần dự đoán, chỉ có trong train.
- Train và test không có giá trị thiếu. ID không bị trùng; ID trong `sample_submission.csv` khớp với test.
- Có 11.689 giao dịch gian lận (1,67%) và 688.271 giao dịch bình thường (98,33%). Hai lớp rất mất cân bằng.
- Nhóm gian lận có trung vị số tiền cao hơn (31,18 USD so với 27,21 USD), thời gian từ giao dịch trước ngắn hơn (3,61 giờ so với 4,73 giờ) và nhiều giao dịch hơn trong 24 giờ gần nhất (4 so với 3). Phân bố hai nhóm vẫn chồng lấn nhiều.
- Gift Cards (7,30%), Crypto Exchange (6,53%) và Gaming (5,51%) có tỷ lệ gian lận cao hơn mức chung. Nhóm No Authentication có tỷ lệ gian lận 6,44%.
- Tỷ lệ gian lận theo giờ và ngày trong tuần chỉ thay đổi nhẹ. Trong các cột số, `cvv_retry_count` có tương quan với `is_fraud` cao nhất (0,212).
- Giao dịch tròn đô la có tỷ lệ gian lận 2,09%, so với 1,67% ở các giao dịch còn lại. Nhóm có thể thử đặc trưng này khi xây mô hình.
- Các kết quả trên mô tả mối liên hệ trong dữ liệu, chưa chứng minh quan hệ nguyên nhân. Cần kiểm tra hiệu quả đặc trưng bằng validation.